# Silver: clean and standardize

Read the Bronze table, trim string values, convert empty strings to null, and remove rows with no business data. CSV columns remain strings until source-specific types and validation rules are agreed.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

CATALOG = "superpower"
BRONZE_TABLE = f"{CATALOG}.bronze.csv_raw"
SILVER_TABLE = f"{CATALOG}.silver.csv_clean"
METADATA_COLUMNS = {"_source_file", "_ingested_at"}

bronze_df = spark.table(BRONZE_TABLE)
business_fields = [
    field for field in bronze_df.schema.fields
    if field.name not in METADATA_COLUMNS
]

def quoted_col(name):
    return F.col(f"`{name.replace('`', '``')}`")

silver_df = bronze_df
for field in business_fields:
    if isinstance(field.dataType, StringType):
        value = F.trim(quoted_col(field.name))
        silver_df = silver_df.withColumn(
            field.name,
            F.when(value == "", F.lit(None)).otherwise(value),
        )

if business_fields:
    has_business_value = F.lit(False)
    for field in business_fields:
        has_business_value = has_business_value | quoted_col(field.name).isNotNull()
    silver_df = silver_df.filter(has_business_value)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
(
    silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Wrote cleaned data to {SILVER_TABLE}")
display(spark.table(SILVER_TABLE).limit(20))